# معمل المخرجات المهيكلة والتحقق عبر Pydantic

أهلاً بك في المعمل العملي للمحاضرة 63!

في المعمل ده هنجرب واحدة من أهم المهارات في بناء أنظمة الـ AI الإنتاجية: **إجبار الموديل على الرد بصيغة بيانات منظمة (Structured Output)** بدل الكلام المرسل:
1. بناء قالب `Movie` باستخدام `Pydantic BaseModel` و `Field`.
2. ربط القالب بالموديل عبر `model.with_structured_output()`.
3. استرجاع الرسالة الخام بجانب القالب (`include_raw=True`) لقراءة التوكنز.
4. بناء قوالب متداخلة (`Nested Schemas`) زي قائمة ممثلين جوه تفاصيل الفيلم!


In [1]:
# تجهيز الموديل وأدوات Pydantic
from typing import List, Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq

load_dotenv()
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("تم تجهيز الموديل و Pydantic بنجاح.")


تم تجهيز الموديل و Pydantic بنجاح.


### 1. بناء قالب الفيلم الأساسي `Movie` وربطه بالموديل

لاحظ إزاي `Field(..., description="...")` بيشرح للموديل يحط إيه في كل خانة، وبيتأكد إن السنة رقم صحيح `int` والتقييم رقم عشري `float`.


In [2]:
class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

structured_llm = model.with_structured_output(Movie)
movie_res = structured_llm.invoke("Provide details about the movie Inception")

print("نوع الكائن الراجع:", type(movie_res).__name__)
print(f"الفيلم: {movie_res.title} | السنة: {movie_res.year} | المخرج: {movie_res.director} | التقييم: {movie_res.rating}")


نوع الكائن الراجع: Movie
الفيلم: Inception | السنة: 2010 | المخرج: Christopher Nolan | التقييم: 8.8


### 2. الاحتفاظ بالرسالة الخام بجانب النتيجة (`include_raw=True`)

لما نستخدم `include_raw=True`، بنحصل على النتيجة المهيكلة `parsed` ومعاها الرسالة الأصلية `raw` عشان نراقب استهلاك التوكنز (`usage_metadata`) وكمان نتأكد لو حصل أي خطأ في التحليل (`parsing_error`).


In [3]:
raw_structured_llm = model.with_structured_output(Movie, include_raw=True)
bundle = raw_structured_llm.invoke("Provide details about the movie Interstellar")

print("البيانات المهيكلة (parsed):", bundle["parsed"])
print("خطأ التحليل (parsing_error):", bundle["parsing_error"])
print("استهلاك التوكنز (usage_metadata):", bundle["raw"].usage_metadata)


البيانات المهيكلة (parsed): title='Interstellar' year=2014 director='Christopher Nolan' rating=8.6
خطأ التحليل (parsing_error): None
استهلاك التوكنز (usage_metadata): {'input_tokens': 159, 'output_tokens': 140, 'total_tokens': 299, 'output_token_details': {'reasoning': 101}}


### 3. القوالب المتداخلة (`Nested Schemas`)

تخيل عايزين نستخرج تفاصيل الفيلم ومعاه قائمة طاقم التمثيل (كل ممثل ليه اسم ودور). هنا بنحط كلاس `Actor` جوه قائمة داخل كلاس `MovieDetails`!


In [4]:
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: List[Actor]
    genres: List[str]
    budget: Optional[float] = Field(None, description="Budget in millions of USD (e.g. 160.0)")

nested_llm = model.with_structured_output(MovieDetails)
details = nested_llm.invoke("Give details for Inception: top 3 actors, genres, and budget in millions.")

print(f"الفيلم: {details.title} ({details.year}) | الميزانية: ${details.budget}M")
print("التصنيفات:", details.genres)
print("طاقم التمثيل:")
for actor in details.cast:
    print(f"  - {actor.name} في دور {actor.role}")


الفيلم: Inception (2010) | الميزانية: $160.0M
التصنيفات: ['Action', 'Science Fiction', 'Thriller']
طاقم التمثيل:
  - Leonardo DiCaprio في دور Dom Cobb
  - Joseph Gordon-Levitt في دور Arthur
  - Elliot Page في دور Ariadne


### 4. جرب بنفسك!

دلوقتي دورك في التجربة الحرة:
1. جرب تغير القالب وتعمل كلاس جديد مثلاً `BookReview` أو `ProductInfo` (اسم المنتج، السعر، المميزات كقائمة `List[str]`).
2. جرب تمرر نص مراجعة منتج غير منظم وشوف الموديل بيستخرج البيانات المهيكلة منه بدقة إزاي!


In [5]:
# جرب بنفسك هنا: استخراج بيانات من مراجعة منتج
class ProductSummary(BaseModel):
    product_name: str
    pros: List[str]
    cons: List[str]
    recommended: bool

review_text = "I bought the UltraBook X1 laptop. The battery lasts 14 hours and the screen is crisp, though the webcam is weak and it gets slightly warm. Overall worth buying!"
extractor = model.with_structured_output(ProductSummary)

extracted = extractor.invoke(f"Extract structured summary from: {review_text}")
print("اسم المنتج:", extracted.product_name)
print("المميزات:", extracted.pros)
print("العيوب:", extracted.cons)
print("هل ينصح به؟", extracted.recommended)


اسم المنتج: UltraBook X1
المميزات: ['battery lasts 14 hours', 'screen is crisp']
العيوب: ['webcam is weak', 'gets slightly warm']
هل ينصح به؟ True
